In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_exposure_adjustment_period_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ExposureAdjustmentPeriodId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "ExposureAdjustmentPeriodKey string,"
        #     "ExposureKey string,"
        #     "ExposureUnit decimal(5,0),"
        #     "ExpsosureType decimal(2,0),"
        #     "ExposureSN decimal(6,0),"
        #     "RecordSN decimal(3,0),"
        #     "AdjustedRequestedInstallmentFrom decimal(4,0),"
        #     "AdjustedRequestedInstallmentTo decimal(4,0),"
        #     "AdjustedFixedInstallmentAmount decimal(15,2),"
        #     "Timestamp timestamp,"
        #     "TransactionDate timestamp,"
        #     "TransactionUnit decimal(5,0),"
        #     "TransactionUser string,"
        #     "ZeroInterestCode string,"
        #     "ZeroInterest string,"
        #     "IsActiveFlag string,"
        #     "UpdatedBy string,"
        #     "UpdatedUnit decimal(5,0),"
        #     "LastUpdatedDate timestamp,"
        #     "LastUpdatedDatetime timestamp,"
        #     "MaxInterestControlFlag string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExposureAdjustmentPeriod"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "profits",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== profits_loan_adjust_period ==================
        profits_loan_adjust_period_extra_filters = None

        if profits_loan_adjust_period_extra_filters is not None:
            profits_loan_adjust_period_extra_filters.append(date_filter)
            profits_loan_adjust_period_final_filters = " AND ".join(
                profits_loan_adjust_period_extra_filters
            )
        else:
            profits_loan_adjust_period_final_filters = date_filter

            profits_loan_adjust_period_selected_fields = [
                "LNS_OPEN_UNIT",
                "LNS_TYPE",
                "LNS_SN",
                "RECORD_SN",
                "ADJ_REQ_INST_FROM",
                "ADJ_REQ_INST_TO",
                "ADJ_FIXED_INST_AMN",
                "TMSTAMP",
                "TRX_DATE",
                "TRX_UNIT",
                "TRX_USR",
                "ZERO_INT_FLG",
                "ENTRY_STATUS",
                "LAST_UPDATE_USR",
                "LAST_UPDATE_UNIT",
                "LAST_UPDATE_DATE",
                "LST_UPD_TMSTAMP",
                "MAX_INST_CONTROL_FLG",
                "BusinessDate",
            ]

            profits_loan_adjust_period = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "profits_loan_adjust_period",
                profits_loan_adjust_period_final_filters,
                profits_loan_adjust_period_selected_fields,
            )

        # ================== profits_loan_details ==================
        profits_loan_details_extra_filters = None

        if profits_loan_details_extra_filters is not None:
            profits_loan_details_extra_filters.append(date_filter)
            profits_loan_details_final_filters = " AND ".join(
                profits_loan_details_extra_filters
            )
        else:
            profits_loan_details_final_filters = date_filter

            profits_loan_details_selected_fields = [
                "ACCOUNT_NUMBER",
                "prft_acc_sn",
                "prft_acc_unit",
                "prft_acc_type",
                "BusinessDate",
            ]

            profits_loan_details = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "profits_loan_details",
                profits_loan_details_final_filters,
                profits_loan_details_selected_fields,
            )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== Joins ==================

        profits_loan_adjust_period_profits_loan_details_join_condition = (
            (
                profits_loan_adjust_period["LNS_SN"]
                == profits_loan_details["prft_acc_sn"]
            )
            & (
                profits_loan_adjust_period["LNS_TYPE"]
                == profits_loan_details["prft_acc_type"]
            )
            & (
                profits_loan_adjust_period["LNS_OPEN_UNIT"]
                == profits_loan_details["prft_acc_unit"]
            )
        )

        loan_details_final_portfolio_join_condition = (
            profits_loan_details["prft_acc_unit"] == portfolio["unit_code_profits"]
        )

        target_table_profits = profits_loan_adjust_period.join(
            profits_loan_details,
            profits_loan_adjust_period_profits_loan_details_join_condition,
            "inner",
        ).join(
            portfolio,
            loan_details_final_portfolio_join_condition,
            "left",
        )

        # ================== Hashkey and Concatenation ==================

        target_table_profits_primary_key_fields = [
            profits_loan_adjust_period.LNS_OPEN_UNIT,
            profits_loan_adjust_period.LNS_TYPE,
            profits_loan_adjust_period.LNS_SN,
            profits_loan_adjust_period.RECORD_SN,
            lit("profits"),
        ]

        target_table_profits_key_fields = [
            profits_loan_adjust_period.LNS_OPEN_UNIT,
            profits_loan_adjust_period.LNS_TYPE,
            profits_loan_adjust_period.LNS_SN,
            profits_loan_adjust_period.RECORD_SN,
        ]

        target_table_profits = (
            target_table_profits.withColumn(
                "ExposureAdjustmentPeriodId",
                sha2(
                    concat_ws(";", *target_table_profits_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "ExposureAdjustmentPeriodKey",
                concat_ws(";", *target_table_profits_key_fields),
            )
            .withColumn(
                "MaxInterestControlFlag",
                when(profits_loan_adjust_period["MAX_INST_CONTROL_FLG"] == 0, 0)
                .when(profits_loan_adjust_period["MAX_INST_CONTROL_FLG"] == " ", 1)
                .otherwise(None),
            )
        )

        result_df = target_table_profits.select(
            target_table_profits.ExposureAdjustmentPeriodId.cast("string"),
            lit("profits").cast("String").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            target_table_profits.ExposureAdjustmentPeriodKey.cast("String"),
            profits_loan_details.ACCOUNT_NUMBER.alias("ExposureKey"),
            target_table_profits.LNS_OPEN_UNIT.cast("decimal(5,0)").alias(
                "ExposureUnit"
            ),
            target_table_profits.LNS_TYPE.cast("decimal(2,0)").alias("ExpsosureType"),
            target_table_profits.LNS_SN.cast("decimal(6,0)").alias("ExposureSN"),
            target_table_profits.RECORD_SN.cast("decimal(3,0)").alias("RecordSN"),
            target_table_profits.ADJ_REQ_INST_FROM.cast("decimal(4,0)").alias(
                "AdjustedRequestedInstallmentFrom"
            ),
            target_table_profits.ADJ_REQ_INST_TO.cast("decimal(4,0)").alias(
                "AdjustedRequestedInstallmentTo"
            ),
            target_table_profits.ADJ_FIXED_INST_AMN.cast("decimal(15,2)").alias(
                "AdjustedFixedInstallmentAmount"
            ),
            target_table_profits.TMSTAMP.cast("timestamp").alias("Timestamp"),
            target_table_profits.TRX_DATE.cast("timestamp").alias("TransactionDate"),
            target_table_profits.TRX_UNIT.cast("decimal(5,0)").alias("TransactionUnit"),
            target_table_profits.TRX_USR.cast("string").alias("TransactionUser"),
            target_table_profits.ZERO_INT_FLG.cast("string").alias("ZeroInterest"),
            target_table_profits.ZERO_INT_FLG.cast("string").alias("ZeroInterestCode"),
            target_table_profits.ENTRY_STATUS.cast("string").alias("IsActiveFlag"),
            target_table_profits.LAST_UPDATE_USR.cast("string").alias("UpdatedBy"),
            target_table_profits.LAST_UPDATE_UNIT.cast("decimal(5,0)").alias(
                "UpdatedUnit"
            ),
            target_table_profits.LAST_UPDATE_DATE.cast("timestamp").alias(
                "LastUpdatedDate"
            ),
            target_table_profits.LST_UPD_TMSTAMP.cast("timestamp").alias(
                "LastUpdatedDatetime"
            ),
            target_table_profits.MaxInterestControlFlag.cast("string"),
            profits_loan_adjust_period.BusinessDate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================
        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_exposure_adjustment_period_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)